In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Store MLflow data locally in mlruns folder
mlflow.set_tracking_uri("file:../mlruns")

# Create experiment
mlflow.set_experiment("credit_risk_model")

print("MLflow setup complete ✅")
print(f"Tracking URI : file:../mlruns")
print(f"Experiment   : credit_risk_model")

MLflow setup complete ✅
Tracking URI : file:../mlruns
Experiment   : credit_risk_model


In [5]:
# Load data
df = pd.read_csv('../data/application_train_processed.csv')

TARGET_COL   = 'TARGET'
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

X = df[FEATURE_COLS]
y = df[TARGET_COL]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,
    random_state = 42
)

# Best parameters from Optuna tuning
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

best_params = {
    'n_estimators'     : 365,
    'max_depth'        : 4,
    'learning_rate'    : 0.06963532425935261,
    'subsample'        : 0.9210667986240065,
    'colsample_bytree' : 0.660632172299131,
    'min_child_weight' : 1,
    'scale_pos_weight' : scale_pos_weight,
    'random_state'     : 42,
    'n_jobs'           : -1
}

print(f"Data loaded     : {X.shape}")
print(f"Train set       : {X_train.shape}")
print(f"Test set        : {X_test.shape}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
print(f"\nBest parameters:")
for k, v in best_params.items():
    print(f"  {k:<25} : {v}")

Data loaded     : (307511, 190)
Train set       : (246008, 190)
Test set        : (61503, 190)
scale_pos_weight: 11.39

Best parameters:
  n_estimators              : 365
  max_depth                 : 4
  learning_rate             : 0.06963532425935261
  subsample                 : 0.9210667986240065
  colsample_bytree          : 0.660632172299131
  min_child_weight          : 1
  scale_pos_weight          : 11.38710976837865
  random_state              : 42
  n_jobs                    : -1


In [6]:
# ── Start MLflow run ────────────────────────────────────────────────────────
with mlflow.start_run(run_name="xgboost_tuned_v1"):

    # ── Step 1: Log parameters ──────────────────────────────────────────────
    mlflow.log_params(best_params)
    mlflow.log_param("train_size", X_train.shape[0])
    mlflow.log_param("test_size",  X_test.shape[0])
    mlflow.log_param("n_features", X_train.shape[1])
    print("Parameters logged ✅")

    # ── Step 2: Train model ─────────────────────────────────────────────────
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train, verbose=False)
    print("Model trained ✅")

    # ── Step 3: Evaluate ────────────────────────────────────────────────────
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred_class = (y_pred_proba >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc  = average_precision_score(y_test, y_pred_proba)

    # KS Statistic
    defaulter_scores = y_pred_proba[y_test == 1]
    repayer_scores   = y_pred_proba[y_test == 0]
    ks_stat = max(
        abs(
            np.searchsorted(np.sort(defaulter_scores), t) / len(defaulter_scores) -
            np.searchsorted(np.sort(repayer_scores),   t) / len(repayer_scores)
        )
        for t in np.linspace(0, 1, 100)
    )

    # Confusion matrix
    cm       = confusion_matrix(y_test, y_pred_class)
    tn, fp, fn, tp = cm.ravel()
    catch_rate = tp / y_test.sum()

    # ── Step 4: Log metrics ─────────────────────────────────────────────────
    mlflow.log_metric("roc_auc",    roc_auc)
    mlflow.log_metric("pr_auc",     pr_auc)
    mlflow.log_metric("ks_stat",    ks_stat)
    mlflow.log_metric("catch_rate", catch_rate)
    mlflow.log_metric("true_positives",  int(tp))
    mlflow.log_metric("false_positives", int(fp))
    mlflow.log_metric("false_negatives", int(fn))
    mlflow.log_metric("true_negatives",  int(tn))
    print("Metrics logged ✅")

    # ── Step 5: Generate and log SHAP plots ─────────────────────────────────
    explainer  = shap.TreeExplainer(model)
    X_sample   = X_test.sample(n=5000, random_state=42)
    shap_values = explainer.shap_values(X_sample)

    # Global bar plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample,
                      plot_type='bar', max_display=20, show=False)
    plt.title('Top 20 Features — Mean Absolute SHAP Value')
    plt.tight_layout()
    plt.savefig('/tmp/shap_bar.png', dpi=150)
    plt.close()
    mlflow.log_artifact('/tmp/shap_bar.png', 'shap_plots')

    # Beeswarm plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample,
                      plot_type='dot', max_display=20, show=False)
    plt.title('SHAP Beeswarm Plot')
    plt.tight_layout()
    plt.savefig('/tmp/shap_beeswarm.png', dpi=150)
    plt.close()
    mlflow.log_artifact('/tmp/shap_beeswarm.png', 'shap_plots')
    print("SHAP plots logged ✅")

    # ── Step 6: Log model ───────────────────────────────────────────────────
    mlflow.xgboost.log_model(
        model,
        artifact_path      = "model",
        registered_model_name = "credit_risk_xgboost"
    )
    print("Model logged ✅")

    # ── Step 7: Log feature list ────────────────────────────────────────────
    feature_df = pd.DataFrame({'feature': FEATURE_COLS})
    feature_df.to_csv('/tmp/features.csv', index=False)
    mlflow.log_artifact('/tmp/features.csv', 'metadata')
    print("Feature list logged ✅")

    # ── Summary ─────────────────────────────────────────────────────────────
    run_id = mlflow.active_run().info.run_id
    print(f"\n{'='*50}")
    print(f"  MLflow Run Complete")
    print(f"{'='*50}")
    print(f"  Run ID   : {run_id}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  PR-AUC   : {pr_auc:.4f}")
    print(f"  KS Stat  : {ks_stat:.4f}")
    print(f"  Catch rate: {catch_rate:.1%}")
    print(f"{'='*50}")

Parameters logged ✅
Model trained ✅
Metrics logged ✅


2026/05/09 18:48:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


SHAP plots logged ✅
Model logged ✅
Feature list logged ✅

  MLflow Run Complete
  Run ID   : 941bc3516d164806b343b74c821e44c2
  ROC-AUC  : 0.7640
  PR-AUC   : 0.2463
  KS Stat  : 0.3936
  Catch rate: 67.8%


Successfully registered model 'credit_risk_xgboost'.
Created version '1' of model 'credit_risk_xgboost'.


In [7]:
import mlflow.pyfunc

# Load model from MLflow registry
model_uri = "models:/credit_risk_xgboost/1"

loaded_model = mlflow.pyfunc.load_model(model_uri)
print(f"Model loaded from MLflow registry ✅")
print(f"  URI : {model_uri}")

# Test prediction on 5 rows
sample = X_test.head(5)
preds  = loaded_model.predict(sample)

print(f"\nTest predictions on 5 rows:")
print(f"  Predictions : {preds}")
print(f"  Actual      : {y_test.head(5).values}")

Model loaded from MLflow registry ✅
  URI : models:/credit_risk_xgboost/1

Test predictions on 5 rows:
  Predictions : [0 0 1 0 1]
  Actual      : [0 0 0 0 0]


✅ Experiment created  : credit_risk_model
✅ Parameters logged   : all 9 hyperparameters
✅ Metrics logged      : ROC-AUC, PR-AUC, KS, catch rate
✅ Artifacts logged    : SHAP plots, feature list
✅ Model registered    : credit_risk_xgboost version 1
✅ Model reloaded      : predictions working